# <font color='blue'>Projeto de Deploy com SageMaker, Lambda e API Gateway</font>
Acompanhe as instruções no capítulo do curso.

In [ ]:
# Imports
import os
import boto3
import pandas as pd
import sagemaker
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

## Configurações Iniciais no SageMaker

In [ ]:
# Define a região
region = boto3.Session().region_name
print(region)

In [ ]:
# Define a role
role = get_execution_role()
print(role)

In [ ]:
# Cria sessão SageMaker
session = sagemaker.Session()
print(session)

## Carregando os Dados

In [ ]:
# Cria o cliente s3
s3 = boto3.client("s3")

In [ ]:
# Download do arquivo do bucket para o ambiente local do SageMaker
s3.download_file(os.environ['SAGEMAKER_BUCKET'], "dataset.csv", "dados_originais.csv")

In [ ]:
# Carrega o arquivo como dataframe do pandas
df = pd.read_csv("./dados_originais.csv")

## Exploração Inicial dos Dados

In [ ]:
# Visualiza as primeiras linhas
df.head()

In [ ]:
# Shape
df.shape

In [ ]:
# Total de valores ausentes
df.isna().sum()

In [ ]:
# Total de duplicatas (complete cases)
df.duplicated().sum()

In [ ]:
# Info
df.info()

In [ ]:
# Total de valores únicos
df.nunique()

In [ ]:
# Resumo estatístico
df.describe()

## Limpeza, Codificação e Preparação dos Dados

In [ ]:
# Colunas
df.columns

In [ ]:
print("Categorias na Variável 'species': ", df['species'].unique())

In [ ]:
print("Categorias na Variável 'island': ", df['island'].unique())

In [ ]:
# Vamos fazer o drop de variáveis que não serão relevantes para o projeto
dados_limpos = df.drop(columns = ['id', 'island', 'sex', 'year'])

In [ ]:
# Info
dados_limpos.info()

In [ ]:
# Vamos fazer o drop de valores ausentes
dados_limpos = dados_limpos.dropna(subset = ['bill_length_mm'])

In [ ]:
dados_limpos.isna().sum()

In [ ]:
dados_limpos.head()

In [ ]:
# Label encoding na variável alvo
dados_limpos['species'] = dados_limpos['species'].replace('Adelie', 0)
dados_limpos['species'] = dados_limpos['species'].replace('Gentoo', 1)
dados_limpos['species'] = dados_limpos['species'].replace('Chinstrap', 2)

In [ ]:
# Shuffle (embaralhamento dos dados)
dados_limpos = dados_limpos.sample(frac = 1).reset_index(drop = True)

In [ ]:
dados_limpos.sample(10)

## Divisão dos Dados em Treino e Teste e Armazenamento no S3

In [ ]:
# Garante que somente as colunas do shape do dataframe serão usadas 
dados_limpos.columns = range(dados_limpos.shape[1])

In [ ]:
dados_limpos.sample(5)

In [ ]:
# Divisão em treino e validação com proporção 80/20
dados_treino, dados_valid = train_test_split(dados_limpos, test_size = 0.2, random_state = 42)

In [ ]:
dados_treino.shape

In [ ]:
dados_valid.shape

In [ ]:
# Nome do bucket S3
bucket_name = os.environ['SAGEMAKER_BUCKET']

In [ ]:
# Salvando e enviando o arquivo de treino
dados_treino.to_csv('dados_treino.csv', header = False, index = False)
key = 'training/treino/dados_treino'
url = 's3://{}/{}'.format(bucket_name, key)
boto3.Session().resource('s3').Bucket(bucket_name).Object(key).upload_file('dados_treino.csv')

In [ ]:
# Salvando e enviando o arquivo de validação
dados_valid.to_csv('dados_valid.csv', header = False, index = False)
key = 'training/val/dados_valid'
url = 's3://{}/{}'.format(bucket_name, key)
boto3.Session().resource('s3').Bucket(bucket_name).Object(key).upload_file('dados_valid.csv')

## Construção do Modelo

In [ ]:
# Nome do bucket S3
bucket_name = os.environ['SAGEMAKER_BUCKET']

In [ ]:
# Nome para salvar o modelo
key = 'models/xgbmodel'

In [ ]:
# Local no S3 para salvar o modelo
s3_output_location = f's3://{bucket_name}/{key}'

In [ ]:
# Define a URI da imagem do modelo XGBoost no SageMaker
image_uri = sagemaker.image_uris.retrieve("xgboost", region, version = "1.5-1")

In [ ]:
# Cria o estimador XGBoost
xgbmodel = sagemaker.estimator.Estimator(image_uri = image_uri,
                                             role = role,
                                             instance_count = 1,
                                             instance_type = 'ml.m4.xlarge',
                                             volume_size = 5,
                                             output_path = s3_output_location,
                                             sagemaker_session = session)

In [ ]:
# Configura os hiperparâmetros
xgbmodel.set_hyperparameters(max_depth = 5,
                                 gamma = 4,
                                 min_child_weight = 6,
                                 objective = 'multi:softmax',
                                 num_class = 3,
                                 num_round = 8)

Descrição dos hiperparâmetros configurados no modelo xgbmodel para o algoritmo XGBoost:

**max_depth (5)**: Define a profundidade máxima de cada árvore de decisão no modelo. Controla a complexidade do modelo, evitando que ele se torne excessivamente complexo e, portanto, sujeito a overfitting. Um valor de 5 permite que cada árvore tenha até 5 níveis de profundidade.

**gamma (4)**: Representa o valor mínimo de redução na função de perda para que uma nova divisão em uma árvore seja feita. Um valor maior torna o modelo mais conservador, pois exige maior melhora na função objetivo para criar novas divisões.

**min_child_weight (6)**: Especifica o peso mínimo de somatório de instâncias em um nó filho para que ele seja subdividido. Este parâmetro ajuda a controlar o underfitting. Um valor maior exige que cada nó tenha mais instâncias, resultando em árvores mais simples.

**objective ('multi:softmax')**: Define a função objetivo para o modelo. Aqui, foi configurado como multi:softmax, que é usado para problemas de classificação multiclasse. Ele gera como saída a classe prevista diretamente, em vez de probabilidades.

**num_class (3)**: Indica o número de classes distintas no problema de classificação. Nesse caso, o modelo está configurado para lidar com 3 classes diferentes.
**num_round (8)**: Refere-se ao número de iterações ou rodadas de boosting. Cada rodada adiciona uma nova árvore ao ensemble para melhorar as previsões do modelo. Com 8 rodadas, o modelo será atualizado iterativamente até esse limite.

## Treinamento do Modelo

In [ ]:
# Define o caminho para os dados
uri_treino = 's3://{}/{}'.format(bucket_name, 'training/treino/dados_treino')
uri_validacao = 's3://{}/{}'.format(bucket_name, 'training/val/dados_val')

In [ ]:
uri_treino

In [ ]:
uri_validacao

In [ ]:
# Cria os inputs
input_treino = TrainingInput(s3_data = uri_treino, content_type = 'text/csv')
input_val = TrainingInput(s3_data = uri_validacao, content_type = 'text/csv')

In [ ]:
# Define os canais para os dados
canais_dados = {'train': input_treino, 'validation': input_val}

In [ ]:
%%time

xgbmodel.fit(inputs = canais_dados)

## Deploy do Modelo com SageMaker

In [ ]:
%%time

xgb_predictor = xgbmodel.deploy(initial_instance_count = 1, instance_type = 'ml.m4.xlarge')

Ao executar a linha acima, o modelo treinado (xgbmodel) é implantado em um endpoint gerenciado pelo Amazon SageMaker para inferência em tempo real. Aqui está o que acontece:

**Criação do Endpoint**: O SageMaker cria um endpoint RESTful onde o modelo ficará disponível para receber solicitações de previsão.

**Configuração da Infraestrutura**: É provisionada uma instância do tipo ml.m4.xlarge para hospedar o modelo. Essa instância é dimensionada com base nos requisitos computacionais do modelo.

**Inicialização do Modelo**: O modelo treinado é carregado na instância provisionada, junto com o ambiente necessário para sua execução, como o container do XGBoost.

**Atribuição ao Objeto xgb_predictor**: O objeto xgb_predictor é criado e serve como uma interface para interagir com o endpoint. Ele permite enviar dados para predição e receber as respostas diretamente no ambiente de desenvolvimento.

Este processo disponibiliza o modelo em um ambiente escalável e gerenciado, pronto para realizar inferências em tempo real a partir de requisições enviadas ao endpoint.

Agora vamos criar a API.

# Fim